<a href="https://colab.research.google.com/github/NismaIrfan850/Heart-Disease-Prediction/blob/main/HeartDiseasePrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,Input,LeakyReLU
from tensorflow.keras.losses import MeanSquaredError, BinaryCrossentropy,SparseCategoricalCrossentropy
from tensorflow.keras.activations import sigmoid,relu,linear,softmax
from tensorflow.keras.regularizers import L2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures,StandardScaler
import pandas as pd
from sklearn.metrics import confusion_matrix,classification_report

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"nismairfan","key":"1d33ff94009c0d5dc0ee7fe7b03496ae"}'}

In [3]:
!pip install kaggle

In [4]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
!kaggle datasets download -d fedesoriano/heart-failure-prediction

Dataset URL: https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction
License(s): ODbL-1.0
100% 8.56k/8.56k [00:00<00:00, 18.4MB/s]



In [6]:
!unzip heart-failure-prediction.zip

Archive:  heart-failure-prediction.zip
  inflating: heart.csv               


In [7]:
train=pd.read_csv("heart.csv")
train.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [8]:
x=train.drop(columns=["HeartDisease"])
y=train["HeartDisease"]
one_hot_encoding=pd.get_dummies(x,columns=["Sex","ChestPainType","RestingECG","ExerciseAngina","ST_Slope"])

In [10]:
x_train,x_temp,y_train,y_temp=train_test_split(one_hot_encoding,y,test_size=0.6,random_state=42)
x_cv,x_test,y_cv,y_test=train_test_split(x_temp,y_temp,test_size=0.5,random_state=42)

In [11]:
scaler=StandardScaler()
normalize_train_data=scaler.fit_transform(x_train)
normalize_cv_data=scaler.transform(x_cv)
normalize_test_data=scaler.transform(x_test)

In [12]:
model=Sequential(
    [
        Dense(units=128,activation="relu",kernel_regularizer=L2(0.05),input_shape=(normalize_train_data.shape[1],)),
        Dense(units=64,activation="relu",kernel_regularizer=L2(0.05)),
        Dense(units=32,activation="relu",kernel_regularizer=L2(0.05)),
        Dense(units=1,activation="sigmoid",kernel_regularizer=L2(0.05)),
    ]
)
model.compile(loss=BinaryCrossentropy(),
              optimizer=tf.keras.optimizers.Adam(0.001),
              metrics=["accuracy"])
model.fit(normalize_train_data,y_train,epochs=100,validation_data=(normalize_cv_data,y_cv))

Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.5531 - loss: 8.3325 - val_accuracy: 0.7236 - val_loss: 7.5759
Epoch 2/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7984 - loss: 7.0267 - val_accuracy: 0.8218 - val_loss: 6.3758
Epoch 3/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8365 - loss: 5.9162 - val_accuracy: 0.8364 - val_loss: 5.3571
Epoch 4/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8420 - loss: 4.9750 - val_accuracy: 0.8400 - val_loss: 4.5127
Epoch 5/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8365 - loss: 4.1914 - val_accuracy: 0.8473 - val_loss: 3.8041
Epoch 6/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8474 - loss: 3.5423 - val_accuracy: 0.8509 - val_loss: 3.2179
Epoch 7/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8420 - loss: 3.0048 - val_accuracy: 0.8509 - val_loss: 2.7338
Epoch 8/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8447 - loss: 2.5626 - val_accuracy: 0.8545 - val_loss:

In [13]:
x_cv_prediction=model.predict(normalize_cv_data)
y_predict=(x_cv_prediction>=0.4).astype(int)
print(confusion_matrix(y_cv,y_predict))
print(classification_report(y_cv,y_predict))

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
[[ 97  16]
 [ 15 147]]
              precision    recall  f1-score   support

           0       0.87      0.86      0.86       113
           1       0.90      0.91      0.90       162

    accuracy                           0.89       275
   macro avg       0.88      0.88      0.88       275
weighted avg       0.89      0.89      0.89       275



In [14]:
x_test_prediction=model.predict(normalize_test_data)
y_test_predict=(x_test_prediction>=0.4).astype(int)
print("confusion matrix is given as")
print(confusion_matrix(y_test,y_test_predict))
print("classification report is given as")
print(classification_report(y_test,y_test_predict))

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
confusion matrix is given as
[[102  19]
 [  9 146]]
classification report is given as
              precision    recall  f1-score   support

           0       0.92      0.84      0.88       121
           1       0.88      0.94      0.91       155

    accuracy                           0.90       276
   macro avg       0.90      0.89      0.90       276
weighted avg       0.90      0.90      0.90       276

